# Comparison of urban indicators in Tampere and Helsinki, Finland

The following Jupyter notebook calculates some urban indicators for Tampere and Helsinki.  

In [11]:
import pathlib 
NOTEBOOK_PATH = pathlib.Path().resolve()
DATA_DIRECTORY = NOTEBOOK_PATH / "data"
OUTPUT_DIRECTORY = NOTEBOOK_PATH / "docs"

In [12]:
# Most necessary imports
import json
import os
import geopandas as gpd
import pandas as pd
# import requests
# from datetime import datetime
import matplotlib.pyplot as plt
# import contextily as cx
import networkx as nx
import osmnx as ox

#import street_network_metrics as net

from functools import partial
from geopy.geocoders import Nominatim

os.environ['PROJ_LIB'] = '/path/to/env/share/proj'
#os.environ['GDAL_DATA'] = '/path/to/env/share/'

In [13]:
#ox.settings.overpass_endpoint = 'https://overpass.kumi.systems/api/interpreter'

In [16]:
def get_network(osm_city):
    assert type(osm_city) == str, 'You should provide a string'
    assert osm_city is not None, 'You should provide a city name'
    graph = ox.graph_from_place(osm_city, network_type='all')
    print(type(graph))
    assert graph is not None, 'No graphs returned by the query'
    return osm_city, graph

In [17]:
CITIES = ['Helsinki, Finland', 'Tampere, Finland']
# nets_t = get_networks([])
# for G in nets:
#     print(f)
nets = { c: {'graph': g} for c, g in [get_network(city) for city in CITIES]}

C:\Users\eemeli.pettersson\Anaconda3\envs\geopython\lib\site-packages\osmnx\geocoder.py:110: FutureWarning: The frame.append method is deprecated and will be removed from pandas in a future version. Use pandas.concat instead.
  gdf = gdf.append(_geocode_query_to_gdf(q, wr, by_osmid))


<class 'networkx.classes.multidigraph.MultiDiGraph'>


C:\Users\eemeli.pettersson\Anaconda3\envs\geopython\lib\site-packages\osmnx\geocoder.py:110: FutureWarning: The frame.append method is deprecated and will be removed from pandas in a future version. Use pandas.concat instead.
  gdf = gdf.append(_geocode_query_to_gdf(q, wr, by_osmid))


<class 'networkx.classes.multidigraph.MultiDiGraph'>


In [18]:
print(type(nets))
print(len(nets.keys()))
for key in nets:
    print(nets[key])
    #ox.plot_graph(G)

<class 'dict'>
2
{'graph': <networkx.classes.multidigraph.MultiDiGraph object at 0x0000028B5B6D2D30>}
{'graph': <networkx.classes.multidigraph.MultiDiGraph object at 0x0000028AE65B6340>}


In [19]:
def graphs_to_gdfs(graph_objs):
    assert len(graph_objs.keys()) > 0, 'No graphs to convert'
    for key in graph_objs:
        graph = ox.project_graph(graph_objs[key]['graph'])
        nodes, edges = ox.graph_to_gdfs(graph)
        graph_objs[key]['edges'] = edges
        graph_objs[key]['nodes'] = nodes
    return graph_objs

In [20]:
gdf_dict = graphs_to_gdfs(nets)


In [27]:
crs = gdf_dict[key]['edges'].crs

In [29]:
crs.to_epsg()

32634

In [58]:
# helsinki = pd.DataFrame(gdf_dict[CITIES[0]]['edges']['highway'].value_counts().reset_index())
# tampere = pd.DataFrame(gdf_dict[CITIES[1]]['edges']['highway'].value_counts().reset_index())

cities_edges = [pd.DataFrame(gdf_dict[city]['edges']['highway'].value_counts().reset_index()) for city in CITIES]
#type(res)
for df, city in zip(cities_edges, CITIES):
    df.style.set_table_attributes("style='display:inline'").set_caption('Table name}')
    print(df)

                                      index  highway
0                                   footway   159415
1                                   service    70375
2                                  cycleway    66290
3                               residential    40416
4                                      path    38122
..                                      ...      ...
124            [path, residential, footway]        1
125  [service, unclassified, living_street]        1
126                    [primary, secondary]        1
127                       [trunk, motorway]        1
128  [living_street, unclassified, service]        1

[129 rows x 2 columns]
                                  index  highway
0                              cycleway    26924
1                                  path    26600
2                               service    25124
3                               footway    19467
4                           residential    19259
..                                  ...      .

In [43]:
# def graph_dict_to_strings(gdf_dict, dir_path=None):
#     strings = {}
#     for key in sorted(gdf_dict.keys()):
#         string = ''
#         string += 'City: {key}\n'.format(key=key)
#         string += 'CRS EPSG: {crs}\n'.format(crs=gdf_dict[key]['edges'].crs.to_epsg())
#         string += 'Index: {index}\n'.format(index=gdf_dict[key]['edges'].index.names)
#         string += 'Value counts\n:{val_counts}\n'.format(val_counts=gdf_dict[key]['edges']['highway'].value_counts())
#         strings[key] = string
#     # if dir_path is not None:
#     #     assert type(dir_path) != str or type(dir_path) != pathlib.Path 
#     #     assert type(path) == pathlib.Path
#     #     with open(dir_path / 
#     return strings
        
        

In [42]:
graph_strings = graph_dict_to_strings(gdf_dict)
for key in sorted(graph_strings.keys()):
    print(graph_strings[key])
    print('\n')

City: Helsinki, Finland
CRS EPSG: 32635
Index: ['u', 'v', 'key']
Value counts
:footway                                   159415
service                                    70375
cycleway                                   66290
residential                                40416
path                                       38122
                                           ...  
[path, residential, footway]                   1
[service, unclassified, living_street]         1
[primary, secondary]                           1
[trunk, motorway]                              1
[living_street, unclassified, service]         1
Name: highway, Length: 129, dtype: int64



City: Tampere, Finland
CRS EPSG: 32634
Index: ['u', 'v', 'key']
Value counts
:cycleway                              26924
path                                  26600
service                               25124
footway                               19467
residential                           19259
                                      ..